# Lezione 8 — Casi d'uso reali: la metodologia completa in azione

**Durata stimata: 120 minuti**

## Obiettivi di apprendimento

- Percorrere dall'inizio alla fine un caso reale (ricoveri ospedalieri ripetuti) attraverso le 10 fasi della metodologia.
- Eseguire una pipeline completa (caricamento → pulizia → modellazione → valutazione) su un dataset reale.
- Applicare la pipeline appresa a un nuovo sottoinsieme di dati in modo autonomo.

Questo è l'ultimo notebook di teoria del corso. Non introduce concetti nuovi: riunisce tutto ciò che già hai visto nelle lezioni 1-7 in due cose: (1) un percorso condensato del caso completo dei ricoveri ripetuti per insufficienza cardiaca congestizia (CHF) attraverso le 10 fasi della metodologia di Rollins, e (2) la pipeline di codice completa — caricamento, pulizia, modellazione e valutazione — sul dataset di ricette, concatenata dall'inizio alla fine in un unico posto, esattamente come la eseguirebbe un data scientist in un proprio progetto.

## 1. Il caso CHF completo, le 10 fasi

Ripasso esecutivo — hai già visto ognuna di queste fasi in dettaglio nei capitoli precedenti del manuale. Qui le vedi tutte di seguito, così come sono avvenute nel progetto reale.

| # | Fase | Cosa è successo |
|---|---|---|
| 1 | **Business Understanding** | Un'assicurazione sanitaria, davanti al calo del finanziamento pubblico per i ricoveri ripetuti, si unisce a IBM. Domanda: come assegnare un budget sanitario limitato per massimizzare la qualità dell'assistenza? Si scopre che ~30% dei pazienti riabilitati viene riammesso in 1 anno (50% in 5 anni), e che la CHF è in cima alla lista. Escono 4 requisiti di business da un workshop. |
| 2 | **Analytic Approach** | Si sceglie la classificazione ad albero decisionale: facile da leggere per personale clinico, dà previsione + probabilità. |
| 3 | **Data Requirements** | Coorte definita con 3 criteri di inclusione (ammissione come degente interno, diagnosi principale di CHF durante 1 anno, 6 mesi di copertura precedente) ed esclusione di comorbilità gravi. Formato obiettivo: un record per paziente. |
| 4 | **Data Collection** | Fonti demografiche, cliniche, di copertura, rimborsi e farmaceutiche. Si rimanda il dato farmaceutico (non disponibile) — decisione corretta, non è servito alla fine. |
| 5 | **Data Understanding** | La statistica descrittiva rivela che la definizione iniziale di "ammissione per CHF" non catturava tutti i casi attesi clinicamente. Si torna a Data Collection. |
| 6 | **Data Preparation** | Si definisce la CHF con precisione (codici di diagnosi + guida clinica), si fissa la finestra di ricovero ripetuto a 30 giorni, si aggregano migliaia di record transazionali in un record per paziente. Coorte finale: **2.343 pazienti**. |
| 7 | **Modeling** | Primo albero: 85% accuracy globale ma solo 45% di successi sul "sì". Si regola il costo relativo di classificazione errata: 1:1 (85%/45%), 9:1 (49%/97%, troppi falsi positivi), 4:1 (81% globale, 68% sensibilità, 85% specificità) — miglior equilibrio. |
| 8 | **Evaluation** | La curva ROC conferma che il modello 4:1 dà la massima separazione rispetto alla linea base — modello ottimale. |
| 9 | **Deployment** | Si traduce in un'app per tablet/browser con valutazione del rischio quasi in tempo reale, con formazione del personale e collaborazione con IT/DBA. |
| 10 | **Feedback** | Dirigenti della gestione clinica rivedono i risultati, si fa follow-up dei pazienti con intervento, si confrontano i tassi di ricovero ripetuto prima/dopo (senza gruppo di controllo, per etica) dopo un anno, e si raffina il modello — includendo la possibilità di incorporare finalmente il dato farmaceutico rimandato. |

**Risultato:** medici che hanno potuto dare una migliore assistenza ai loro pazienti incorporando informazioni basate sui dati, quasi in tempo reale, nelle loro decisioni cliniche.

## 2. Il caso gemello: Cognos e diabete giovanile

La stessa metodologia — e la stessa tecnica ad albero decisionale — è stata riutilizzata in un problema clinico diverso: prevedere il rischio di ospedalizzazione di pazienti con diabete giovanile. È stata distribuita come applicazione interattiva con **Cognos**, con tre livelli di vista:

- **Vista nazionale**: mappa interattiva del rischio di ospedalizzazione in tutto il paese.
- **Vista per sottogruppo**: report riassuntivo del rischio dentro un nodo specifico dell'albero decisionale.
- **Vista individuale**: report dettagliato del rischio e della storia clinica di un paziente specifico.

Questo dimostra che la metodologia di Rollins non è una ricetta che serve solo per il caso CHF: è un processo riutilizzabile per qualsiasi problema, persino con la stessa tecnica di modellazione.

## 3. La pipeline completa: ricette, dall'inizio alla fine

Da qui in poi, tutto è codice reale che già conosci dalle lezioni 3, 4 e 5. Lo vedrai concatenato in un unico blocco continuo, con commenti che segnano a quale fase della metodologia corrisponde ogni parte — esattamente come lo eseguirebbe un data scientist nel proprio progetto, senza pause tra le fasi.

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np
import re
import random
from sklearn import tree
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import itertools

### Fase: Data Collection

Scarica il CSV direttamente dal server di IBM e lo carica come DataFrame — esattamente come nella lezione 3.

In [ ]:
# Etapa: Data Collection
recipes = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-DS0103EN-SkillsNetwork/labs/Module%202/recipes.csv"
)

print("Data read into dataframe!", recipes.shape)

### Fase: Data Preparation

Il blocco di pulizia completo visto nella lezione 4: sistemare il nome della colonna, mettere tutto in minuscolo, unificare nomi di cucina incoerenti, eliminare cucine con meno di 50 ricette, e convertire "Yes"/"No" in 1/0.

In [ ]:
# Etapa: Data Preparation

# La prima colonna porta il nome della cucina, ma mal etichettata come "country"
column_names = recipes.columns.values
column_names[0] = "cuisine"
recipes.columns = column_names

# Minuscolo, affinché "Mexican" e "mexican" non contino come cucine distinte
recipes["cuisine"] = recipes["cuisine"].str.lower()

# Corregge nomi di cucina incoerenti
recipes.loc[recipes["cuisine"] == "korea", "cuisine"] = "korean"
recipes.loc[recipes["cuisine"] == "japan", "cuisine"] = "japanese"
recipes.loc[recipes["cuisine"] == "china", "cuisine"] = "chinese"
recipes.loc[recipes["cuisine"] == "thailand", "cuisine"] = "thai"
recipes.loc[recipes["cuisine"] == "india", "cuisine"] = "indian"

# Elimina cucine con meno di 50 ricette: non ci sono abbastanza esempi per imparare un pattern affidabile
recipes_counts = recipes["cuisine"].value_counts()
cuisines_indices = recipes_counts > 50
cuisines_to_keep = list(np.array(recipes_counts.index.values)[np.array(cuisines_indices)])
recipes = recipes.loc[recipes["cuisine"].isin(cuisines_to_keep)]

# Converte "Yes"/"No" in 1/0, il formato di cui scikit-learn ha bisogno
recipes = recipes.replace(to_replace="Yes", value=1)
recipes = recipes.replace(to_replace="No", value=0)

print("Dataset limpio y listo:", recipes.shape)

### Fase: Modeling

Sottoinsieme di 5 cucine (coreana, giapponese, cinese, thailandese e indiana), divisione train/test con `sample_n = 30` e `random.seed(1234)` affinché il risultato sia riproducibile, e addestramento di un `DecisionTreeClassifier` con `max_depth=15` — come nella lezione 5.

In [ ]:
# Etapa: Modeling
bamboo = recipes[recipes.cuisine.isin(["korean", "japanese", "chinese", "thai", "indian"])]

sample_n = 30
random.seed(1234)  # fissa il seme casuale affinché il risultato sia riproducibile

# Insieme di test: 30 ricette per ogni cucina
bamboo_test = bamboo.groupby("cuisine", group_keys=False).apply(lambda x: x.sample(sample_n))
bamboo_test_ingredients = bamboo_test.iloc[:, 1:]
bamboo_test_cuisines = bamboo_test["cuisine"]

# Insieme di addestramento: tutto ciò che NON è finito nell'insieme di test
bamboo_train_index = bamboo.index.isin(bamboo_test.index)
bamboo_train = bamboo[~bamboo_train_index]
bamboo_train_ingredients = bamboo_train.iloc[:, 1:]
bamboo_train_cuisines = bamboo_train["cuisine"]

bamboo_train_tree = tree.DecisionTreeClassifier(max_depth=15)
bamboo_train_tree.fit(bamboo_train_ingredients, bamboo_train_cuisines)

print("Decision tree model saved to bamboo_train_tree!")

Puoi visualizzare l'albero addestrato con `plot_tree` di scikit-learn, esattamente come nella lezione 5 — non usiamo `graphviz`, perché non funziona su Google Colab senza installare dipendenze extra.

In [ ]:
# Visualizzazione dell'albero con plot_tree (non graphviz), come nella lezione 5
plt.figure(figsize=(20, 10))
tree.plot_tree(
    bamboo_train_tree,
    feature_names=list(bamboo_train_ingredients.columns.values),
    class_names=sorted(bamboo_train_cuisines.unique()),
    filled=True,
    fontsize=6,
    max_depth=3,  # mostriamo solo i primi livelli affinché sia leggibile
)
plt.show()

In [ ]:
# .predict() usa il modello già addestrato per prevedere la cucina
# di ricette che il modello NON ha mai visto durante l'addestramento
bamboo_pred_cuisines = bamboo_train_tree.predict(bamboo_test_ingredients)

bamboo_pred_cuisines[:10]

### Fase: Evaluation

Matrice di confusione (righe = cucina reale, colonne = cucina prevista) e accuracy globale — la stessa lettura fatta con la tabella dei modelli del caso CHF: non fermarti mai a un solo numero.

In [ ]:
# Etapa: Evaluation
test_cuisines = np.unique(bamboo_test_cuisines)
bamboo_confusion_matrix = confusion_matrix(bamboo_test_cuisines, bamboo_pred_cuisines, labels=test_cuisines)

title = "Bamboo Confusion Matrix"
cmap = plt.cm.Blues

plt.figure(figsize=(8, 6))
bamboo_confusion_matrix_pct = (
    bamboo_confusion_matrix.astype("float") / bamboo_confusion_matrix.sum(axis=1)[:, np.newaxis]
) * 100

plt.imshow(bamboo_confusion_matrix_pct, interpolation="nearest", cmap=cmap)
plt.title(title)
plt.colorbar()
tick_marks = np.arange(len(test_cuisines))
plt.xticks(tick_marks, test_cuisines)
plt.yticks(tick_marks, test_cuisines)

fmt = ".2f"
thresh = bamboo_confusion_matrix_pct.max() / 2.0
for i, j in itertools.product(range(bamboo_confusion_matrix_pct.shape[0]), range(bamboo_confusion_matrix_pct.shape[1])):
    plt.text(
        j, i, format(bamboo_confusion_matrix_pct[i, j], fmt),
        horizontalalignment="center",
        color="white" if bamboo_confusion_matrix_pct[i, j] > thresh else "black",
    )

plt.tight_layout()
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.show()

In [ ]:
# Accuracy globale: proporzione di previsioni corrette sul totale delle ricette di test
accuracy_global = accuracy_score(bamboo_test_cuisines, bamboo_pred_cuisines)
print(f"Accuracy global del modelo: {accuracy_global:.2%}")

## Riepilogo: lo stesso viaggio, due modi di percorrerlo

Nota la simmetria tra ciò che hai appena eseguito e il caso CHF visto all'inizio di questo notebook:

| Fase della metodologia | Caso CHF | Caso ricette (questo notebook) |
|---|---|---|
| Data Collection | Fonti demografiche, cliniche, di copertura, rimborsi, farmaceutiche | `pd.read_csv()` dal server di IBM |
| Data Preparation | Definire la CHF con precisione, finestra di 30 giorni, aggregare a 1 record/paziente (2.343 pazienti) | Nome della colonna, minuscolo, coerenza, filtro <50, Yes/No → 1/0 |
| Modeling | Albero decisionale, regolazione del costo relativo di classificazione errata (1:1, 9:1, 4:1) | `DecisionTreeClassifier(max_depth=15)` su 5 cucine |
| Evaluation | Curva ROC — il modello 4:1 dà la massima separazione | Matrice di confusione + accuracy globale |

Il caso CHF non ha dati pubblici che tu possa eseguire, ma la logica appena eseguita sulle ricette è esattamente la stessa eseguita da quel team di IBM su dati ospedalieri reali. Ora vai al notebook di pratica di questa lezione e ripeti questa stessa pipeline tu da solo, su un sottoinsieme di cucine europee — con meno indizi rispetto a qualsiasi pratica precedente del corso.